# 02 Extraction PDF, sections et tableaux

**Objectif** : examiner les PDF placés dans `data/raw/`, comparer les parseurs et vérifier la tracabilité jusqu'à  la cellule.

**Critère de passage** : le parseur choisit une sortie exploitable ; les tableaux critiques sont revus avant indexation.

In [4]:
from pathlib import Path
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table, module_available

ROOT = bootstrap()
RAW_DIR = ROOT / 'data' / 'raw'
pdf_files = sorted(RAW_DIR.glob('*.pdf'))
{
    'raw_directory': str(RAW_DIR),
    'pdf_count': len(pdf_files),
    'pymupdf': module_available('pymupdf') or module_available('fitz'),
    'pdfplumber': module_available('pdfplumber'),
    'camelot': module_available('camelot'),
    'docling': module_available('docling'),
}

{'raw_directory': 'C:\\Users\\choun\\Downloads\\Prudential_Evidence_Lab_MVP_Source\\prudential_evidence_lab\\data\\raw',
 'pdf_count': 1,
 'pymupdf': True,
 'pdfplumber': False,
 'camelot': False,
 'docling': True}

In [5]:
page_rows = []
if pdf_files and (module_available('pymupdf') or module_available('fitz')):
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    with fitz.open(pdf_files[0]) as document:
        for page_index in range(len(document)):
            page = document[page_index]
            text = page.get_text('text')
            page_rows.append({
                'page': page_index + 1,
                'characters': len(text),
                'blocks': len(page.get_text('blocks')),
                'potential_scan': len(text.strip()) < 40,
            })
else:
    print('Placer un PDF public dans data/raw/ et installer .[ingestion].')
display_table(page_rows)

,page,characters,blocks,potential_scan
0,1,2101,46,False
1,2,1929,44,False
2,3,2624,49,False
3,4,1383,39,False
4,5,706,23,False
5,6,4013,51,False
6,7,2271,23,False
7,8,823,16,False
8,9,3098,47,False
9,10,2577,45,False


In [6]:
table_rows = []
if pdf_files and module_available('pdfplumber'):
    import pdfplumber
    with pdfplumber.open(pdf_files[0]) as document:
        for page_index, page in enumerate(document.pages, start=1):
            for table_index, table in enumerate(page.extract_tables() or [], start=1):
                table_rows.append({
                    'page': page_index,
                    'table': table_index,
                    'rows': len(table),
                    'columns': max((len(row) for row in table), default=0),
                    'preview': str(table[:2])[:180],
                })
display_table(table_rows)

# pdfplumber est ici un diagnostic exploratoire : le nombre de lignes qu'il reconstruit varie selon sa version. Le contrat est vérifié sur les cellules.

from ingestion.pymupdf_fallback import extract_qrt_coverage_table

if pdf_files:
    qrt_table, qrt_warnings = extract_qrt_coverage_table(
        pdf_files[0], entity='Groupe Foyer', period='2025'
    )
    assert qrt_table is not None, f'Extraction QRT impossible : {qrt_warnings}'
    extracted = {cell.row_code: cell for cell in qrt_table.cells}
    assert set(extracted) == {'R0660', 'R0680', 'R0690'}
    assert all(cell.column_code == 'C0010' for cell in extracted.values())
    assert extracted['R0660'].normalized_value == 2407647.0
    assert extracted['R0680'].normalized_value == 840040.0
    assert extracted['R0690'].normalized_value == 2.87
    display_table([cell.model_dump() for cell in qrt_table.cells])

In [7]:
from app.store.artifacts import store

localized_cells = [
    {
        'document': chunk.document_id,
        'page': chunk.locator.page,
        'table': chunk.locator.table_id,
        'row': chunk.locator.row,
        'column': chunk.locator.column,
        'bbox': chunk.locator.bbox,
    }
    for chunk in store.chunks if chunk.locator.table_id
]
qrt_cells = [cell for cell in localized_cells if cell['document'] == 'foyer_group_qrt_2025']
assert {cell['row'] for cell in qrt_cells} == {'R0660', 'R0680', 'R0690'}
assert all(cell['column'] == 'C0010' and cell['page'] == 7 for cell in qrt_cells)
display_table(localized_cells)

,document,page,table,row,column,bbox
0,foyer_group_qrt_2025,7,S.23.01.22,R0660,C0010,"[495.6, 270.13, 564.4, 276.85]"
1,foyer_group_qrt_2025,7,S.23.01.22,R0680,C0010,"[495.6, 278.41, 564.37, 285.13]"
2,foyer_group_qrt_2025,7,S.23.01.22,R0690,C0010,"[495.6, 286.93, 564.33, 293.65]"
3,table_extraction_demo,1,demo-table-1,2,3,"[120.0, 240.0, 260.0, 278.0]"
